# Building Mini-GPT Architecture

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
vocab_size = 28

d_model = 32
context_length = 40
num_heads = 4
ffn_dim = 128

print("TensorFlow version:", tf.__version__)
print("Vocabulary size:", vocab_size)
print("Embedding dimension:", d_model)
print("Number of attention heads:", num_heads)
print("Dimension per head:", d_model // num_heads)

TensorFlow version: 2.21.0
Vocabulary size: 28
Embedding dimension: 32
Number of attention heads: 4
Dimension per head: 8


In [3]:
token_embedding = layers.Embedding(
    input_dim=vocab_size,
    output_dim=d_model
)

position_embedding = layers.Embedding(
    input_dim=context_length,
    output_dim=d_model
)

# Example batch: 2 sequences, each containing 40 token IDs
dummy_tokens = tf.random.uniform(
    shape=(2, context_length),
    minval=0,
    maxval=vocab_size,
    dtype=tf.int32
)

positions = tf.range(context_length)

token_vectors = token_embedding(dummy_tokens)
position_vectors = position_embedding(positions)

x = token_vectors + position_vectors

print("Token IDs shape:", dummy_tokens.shape)
print("Token embeddings shape:", token_vectors.shape)
print("Position embeddings shape:", position_vectors.shape)
print("Combined embeddings shape:", x.shape)

Token IDs shape: (2, 40)
Token embeddings shape: (2, 40, 32)
Position embeddings shape: (40, 32)
Combined embeddings shape: (2, 40, 32)


In [4]:
attention_layer = layers.MultiHeadAttention(
    num_heads = num_heads,
    key_dim = d_model//num_heads
)

attention_output = attention_layer(
    query = x,
    key = x,
    value = x,
    use_causal_mask = True
)

print("Input Shape: ", x.shape)
print("Attention Output Shape: ",attention_output.shape)

Input Shape:  (2, 40, 32)
Attention Output Shape:  (2, 40, 32)


In [5]:
# Residual connection + LayerNorm after attention
residual_1 = layers.Add()([x, attention_output])
norm_1 = layers.LayerNormalization(epsilon=1e-6)(residual_1)

print("Residula Layer Shape: ", residual_1.shape)
print("Normalized Layer Shape: ",norm_1.shape)

Residula Layer Shape:  (2, 40, 32)
Normalized Layer Shape:  (2, 40, 32)


In [6]:

ffn = keras.Sequential([
    layers.Dense(ffn_dim, activation="relu"),
    layers.Dense(d_model)
])

ffn_output = ffn(norm_1)

print("FFN Input Shape : ",norm_1.shape)
print("FFN Output Shape : ",ffn_output.shape)

FFN Input Shape :  (2, 40, 32)
FFN Output Shape :  (2, 40, 32)


In [7]:
residual_2 = layers.Add()([norm_1, ffn_output])
block_output = layers.LayerNormalization(epsilon=1e-6)(residual_2)

print("Transformer block output:", block_output.shape)

Transformer block output: (2, 40, 32)


In [8]:
# Project each token representation to vocabulary scores
lm_head = layers.Dense(vocab_size)

logits = lm_head(block_output)

print("Transformer block output shape: ",block_output.shape)
print("Logits Shape: ", logits.shape)

Transformer block output shape:  (2, 40, 32)
Logits Shape:  (2, 40, 28)


In [9]:
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ffn_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model//num_heads
        )

        self.ffn = keras.Sequential([
            layers.Dense(ffn_dim, activation="relu"),
            layers.Dense(d_model)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self,x, training=False):
        attention_output = self.attention(
            query=x,
            key=x,
            value=x,
            use_causal_mask=True,
            training=training
        )

        x = self.norm1(
            x + self.dropout1(attention_output, training=training)
        )

        ffn_output = self.ffn(x, training=training)

        x = self.norm2(
            x + self.dropout2(ffn_output, training=training)
        )

        return x

In [10]:
block =TransformerBlock(
    d_model=d_model,
    num_heads=num_heads,
    ffn_dim=ffn_dim
)

test_block = block(x, training=False)

print("Input Shape: ",x.shape)
print("Block Output Shape: ",test_block.shape)

Input Shape:  (2, 40, 32)
Block Output Shape:  (2, 40, 32)


In [11]:
class MiniGPT(keras.Model):
    def __init__(
        self,
        vocab_size,
        context_length,
        d_model,
        num_heads,
        ffn_dim,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = layers.Embedding(
            vocab_size, d_model
        )

        self.position_embedding = layers.Embedding(
            context_length, d_model
        )

        self.transformer_block = TransformerBlock(
            d_model=d_model,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            dropout=dropout
        )

        self.final_norm = layers.LayerNormalization(epsilon=1e-6)
        self.lm_head = layers.Dense(vocab_size)

    def call(self, token_ids, training=False):
        seq_len = tf.shape(token_ids)[1]

        position = tf.range(seq_len)

        x = self.token_embedding(token_ids)
        x = x + self.position_embedding(position)

        x = self.transformer_block(x, training=False)
        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

In [12]:
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    d_model=d_model,
    num_heads=num_heads,
    ffn_dim=ffn_dim
)

test_tokens = tf.random.uniform(
    shape = (2,context_length),
    minval = 0,
    maxval = vocab_size,
    dtype=tf.int32
)

test_logits = model(test_tokens, training=False)

print("Input shape:", test_tokens.shape)
print("Output shape:", test_logits.shape)
print("Total parameters:", model.count_params())

Input shape: (2, 40)
Output shape: (2, 40, 28)
Total parameters: 15868


In [13]:
model.summary()

Model: "mini_gpt"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (2, 40, 32)                 │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_3 (Embedding)              │ (40, 32)                    │           1,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block_1                  │ ?                           │          12,704 │
│ (TransformerBlock)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer_normalization_6                │ (2, 40, 32)                 │              64 │
│ (LayerNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (2, 40, 28)                 │             924 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 15,868 (61.98 KB)

 Trainable params: 15,868 (61.98 KB)

 Non-trainable params: 0 (0.00 B)

# Train Your Mini GPT with TensorFlow

In [14]:
from pathlib import Path

url = (
    "https://storage.googleapis.com/"
    "download.tensorflow.org/data/shakespeare.txt"
)

text_path = tf.keras.utils.get_file(
    fname="shakespeare.txt",
    origin=url
)

text = Path(text_path).read_text(encoding="utf-8")

print("Dataset path:", text_path)
print("Text length:", len(text), "characters")
print("Unique characters:", len(set(text)))
print("\nFirst 300 characters:\n")
print(text[:300])

Dataset path: C:\Users\DELL\.keras\datasets\shakespeare.txt
Text length: 1115394 characters
Unique characters: 65

First 300 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [15]:
vocab = sorted(set(text))
vocab_size = len(vocab)

char_to_id = {char: idx for idx, char in enumerate(vocab)}
id_to_char = {idx: char for idx, char in enumerate(vocab)}

encoded_text = [char_to_id[char] for char in text]

print("Vocabulary size:", vocab_size)
print("Encoded token count:", len(encoded_text))
print("First 30 token IDs:", encoded_text[:30])

decoded_text = "".join(id_to_char[i] for i in encoded_text)
print("Encoding/decoding verified:", decoded_text == text)

Vocabulary size: 65
Encoded token count: 1115394
First 30 token IDs: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43]
Encoding/decoding verified: True


In [16]:
context_length = 40
batch_size = 32

# Split the text chronologically: 90% training, 10% validation
split_index = int(len(encoded_text)*0.90)

train_tokens = np.array(
    encoded_text[:split_index],
    dtype=np.int32
)

val_tokens = np.array(
    encoded_text[split_index:],
    dtype=np.int32
)

print("Training Characters: ",len(train_tokens))
print("Validation Characters: ",len(val_tokens))
print("context Length: ",context_length)
print("Batch Size: ",batch_size)

Training Characters:  1003854
Validation Characters:  111540
context Length:  40
Batch Size:  32


In [17]:
def create_dataset(tokens, context_length=40, batch_size=32, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices(tokens)

    dataset = dataset.window(
        context_length + 1,
        shift=1,
        drop_remainder = True
    )

    dataset = dataset.flat_map(
        lambda window: window.batch(context_length + 1)
    )

    dataset = dataset.map(
        lambda window: (window[:-1], window[1:]),
        num_parallel_calls = tf.data.AUTOTUNE
    )

    if shuffle: 
        dataset = dataset.shuffle(
            10_000,
            reshuffle_each_iteration = True
        )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_dataset = create_dataset(
    train_tokens,
    context_length,
    batch_size,
    shuffle=True
)

val_dataset = create_dataset(
    val_tokens,
    context_length,
    batch_size
)

# Inspect one batch
for x_batch, y_batch in train_dataset.take(1):
    print("Input batch shape:", x_batch.shape)
    print("Target batch shape:", y_batch.shape)
    print("\nInput example:")
    print("".join(id_to_char[int(i)] for i in x_batch[0].numpy()))
    print("\nTarget example:")
    print("".join(id_to_char[int(i)] for i in y_batch[0].numpy()))
    

Input batch shape: (32, 40)
Target batch shape: (32, 40)

Input example:
izen:
What he cannot help in his nature,

Target example:
zen:
What he cannot help in his nature, 


In [18]:
vocab_size = len(vocab)  # 65
context_length = 40
d_model = 32
num_heads = 4
ffn_dim = 128
dropout_rate = 0.1

print("Vocabulary size:", vocab_size)
print("Context length:", context_length)
print("Embedding dimension:", d_model)
print("Attention heads:", num_heads)
print("Feed-forward dimension:", ffn_dim)

Vocabulary size: 65
Context length: 40
Embedding dimension: 32
Attention heads: 4
Feed-forward dimension: 128


In [19]:
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ffn_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model//num_heads
        )

        self.ffn = keras.Sequential([
            layers.Dense(ffn_dim, activation="relu"),
            layers.Dense(d_model)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self,x, training=False):
        attention_output = self.attention(
            query=x,
            key=x,
            value=x,
            use_causal_mask=True,
            training=training
        )

        x = self.norm1(
            x + self.dropout1(attention_output, training=training)
        )

        ffn_output = self.ffn(x, training=training)

        x = self.norm2(
            x + self.dropout2(ffn_output, training=training)
        )

        return x

In [20]:
block =TransformerBlock(
    d_model=d_model,
    num_heads=num_heads,
    ffn_dim=ffn_dim
)

test_block = block(x, training=False)

print("Input Shape: ",x.shape)
print("Block Output Shape: ",test_block.shape)

Input Shape:  (2, 40, 32)
Block Output Shape:  (2, 40, 32)


In [21]:
class MiniGPT(keras.Model):
    def __init__(
        self,
        vocab_size,
        context_length,
        d_model,
        num_heads,
        ffn_dim,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = layers.Embedding(
            vocab_size, d_model
        )

        self.position_embedding = layers.Embedding(
            context_length, d_model
        )

        self.transformer_block = TransformerBlock(
            d_model=d_model,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            dropout=dropout
        )

        self.final_norm = layers.LayerNormalization(epsilon=1e-6)
        self.lm_head = layers.Dense(vocab_size)

    def call(self, token_ids, training=False):
        seq_len = tf.shape(token_ids)[1]

        position = tf.range(seq_len)

        x = self.token_embedding(token_ids)
        x = x + self.position_embedding(position)

        x = self.transformer_block(x, training=training)
        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

In [22]:
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    d_model=d_model,
    num_heads=num_heads,
    ffn_dim=ffn_dim
)

test_tokens = tf.random.uniform(
    shape = (2,context_length),
    minval = 0,
    maxval = vocab_size,
    dtype=tf.int32
)

test_logits = model(test_tokens, training=False)

print("Input shape:", test_tokens.shape)
print("Output shape:", test_logits.shape)
print("Total parameters:", model.count_params())

Input shape: (2, 40)
Output shape: (2, 40, 65)
Total parameters: 18273


In [23]:
model.summary()

Model: "mini_gpt_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)              │ (2, 40, 32)                 │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_5 (Embedding)              │ (40, 32)                    │           1,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block_3                  │ ?                           │          12,704 │
│ (TransformerBlock)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer_normalization_11               │ (2, 40, 32)                 │              64 │
│ (LayerNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ (2, 40, 65)                 │           2,145 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 18,273 (71.38 KB)

 Trainable params: 18,273 (71.38 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
import os

os.makedirs("checkpoints", exist_ok=True)

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss= keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["sparse_categorical_accuracy"]
)

print("Model Compiled Successfully!")

Model Compiled Successfully!


In [25]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="checkpoints/best.weights.h5",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1
    )
]
print("Checkpoint and early-stopping callbacks configured.")

Checkpoint and early-stopping callbacks configured.


In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    verbose=1,
    callbacks=callbacks
)

Epoch 1/5


C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


   3474/Unknown 66s 18ms/step - loss: 2.0726 - sparse_categorical_accuracy: 0.3910